## Importing libraries

In [2]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## uploading dataset

In [4]:
df=pd.read_csv("/content/fake_news_dataset.csv")
df

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake
...,...,...,...,...,...,...,...
15832,Employee control program.,by couple home case opportunity first fear the...,2023-04-06,CNN,Kaitlyn Adams,Science,real
15833,Serious maybe determine.,painting stand store establish economic recent...,2024-03-01,CNN,Amber Patterson,Science,real
15834,Key Congress from course.,system position would fact man imagine receive...,2023-05-05,Global Times,Michelle Ibarra,Entertainment,fake
15835,Soldier last suddenly we.,run interview find week oil everyone agent sit...,2022-12-14,Reuters,Donald Haney,Business,fake


In [5]:
df.head()

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake


In [7]:

df=df[['text','label']]
df

,text,label
0,more tax development both store agreement lawy...,real
1,probably guess western behind likely next inve...,fake
2,them identify forward present success risk sev...,fake
3,phone which item yard Republican safe where po...,fake
4,wonder myself fact difficult course forget exa...,fake
...,...,...
15831,else floor get away head foreign three intervi...,fake
15832,by couple home case opportunity first fear the...,real
15833,painting stand store establish economic recent...,real
15834,system position would fact man imagine receive...,fake


In [8]:
print(df['label'].value_counts())

label
fake    8024
real    7812
Name: count, dtype: int64


In [40]:
df['label'] = df['label'].map({'fake': 0, 'real': 1})

In [41]:
print(df['label'].head())
print(df['label'].unique())

0    1
1    0
2    0
3    0
4    0
Name: label, dtype: int64
[1 0]


In [43]:
df['label'] = df['label'].astype(int)
df['label'].head()

,label
0,1
1,0
2,0
3,0
4,0


## part 1 : data preprocessing

In [44]:
# Part 1: Data Preprocessing
#- Convert text to lowercase - Remove punctuation - Remove numbers using regex - Remove stopwords - Apply lemmatization or stemming
#converting text to lowerr case
df['text'] = df['text'].str.lower()
print(df['text'].head())

0    tax development store agreement lawyer hear ou...
1    probably guess western behind likely next inve...
2    identify forward present success risk several ...
3    phone item yard republican safe police identif...
4    wonder fact difficult course forget exactly pa...
Name: text, dtype: object


In [45]:
# now removing punctuations
df['text'] = df['text'].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))
print(df['text'].head())

0    tax development store agreement lawyer hear ou...
1    probably guess western behind likely next inve...
2    identify forward present success risk several ...
3    phone item yard republican safe police identif...
4    wonder fact difficult course forget exactly pa...
Name: text, dtype: object


In [46]:
# now removiing numbers if any using regres
df['text'] = df['text'].apply(lambda x: re.sub(r'\d+', '', x))
print(df['text'].head())

0    tax development store agreement lawyer hear ou...
1    probably guess western behind likely next inve...
2    identify forward present success risk several ...
3    phone item yard republican safe police identif...
4    wonder fact difficult course forget exactly pa...
Name: text, dtype: object


In [47]:
# removing stop word
stop_words = set(stopwords.words('english'))
# Remove stopwords
df['text'] = df['text'].apply(
    lambda x: " ".join([word for word in x.split() if word not in stop_words])
)
print(df['text'].head())

0    tax development store agreement lawyer hear ou...
1    probably guess western behind likely next inve...
2    identify forward present success risk several ...
3    phone item yard republican safe police identif...
4    wonder fact difficult course forget exactly pa...
Name: text, dtype: object


In [84]:
lemmatizer = WordNetLemmatizer()

# Apply lemmatization
df['lemmatized'] = df['text'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))
print(df['lemmatized'].head())

13342    effort majority usually meet responsibility jo...
169      candidate available approach light campaign te...
2099     election call success cultural color could sel...
4948     brother conference expect issue pretty team me...
5771     entire like think scientist party task skill o...
Name: lemmatized, dtype: object


## part 2: Text to Numerical Data - Use Tokenizer - Convert text to sequences - Apply padding

In [81]:
# Use Tokenizer -
# Convert text to sequences - Apply padding

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")

# Fit on REAL text data
tokenizer.fit_on_texts(df['lemmatized'])

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(df['lemmatized'])
print("Sequences:")

#  Apply padding
padded_sequences = pad_sequences(sequences, padding='post')

print(padded_sequences)




Sequences:
[[385 745 714 ...   0   0   0]
 [301 101 640 ...   0   0   0]
 [273 672  10 ...   0   0   0]
 ...
 [188 572  16 ...   0   0   0]
 [580  68 376 ...   0   0   0]
 [347 493 828 ...   0   0   0]]


In [78]:
print(padded_sequences.shape)

(15836, 283)


In [85]:
# Define max_len from the padded sequences shape
max_len = padded_sequences.shape[1]

In [86]:
# Define the preprocess function
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])
    return text

##part 3: Neural Network Model -



In [80]:
# Use TensorFlow - Include Embedding layer - Include LSTM or Dense layers


model = Sequential()

# Embedding layer
model.add(Embedding(5000, 128, input_length=100))

# LSTM layer
model.add(LSTM(64))

# Dense layer (output)
model.add(Dense(1, activation='sigmoid'))

# compile model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])





## part  4 testing & training

## Testing

In [52]:
# Split dataset into training and testing - Train model for multiple epochs

X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences,
    df['label'],
    test_size=0.2,
    random_state=42
)


## training

In [53]:
# train model
history = model.fit(X_train,y_train,epochs=5,batch_size=32,validation_data=(X_test, y_test))

Epoch 1/5
396/396 ━━━━━━━━━━━━━━━━━━━━ 73s 171ms/step - accuracy: 0.4934 - loss: 0.6934 - val_accuracy: 0.5054 - val_loss: 0.6931
Epoch 2/5
396/396 ━━━━━━━━━━━━━━━━━━━━ 78s 162ms/step - accuracy: 0.5001 - loss: 0.6935 - val_accuracy: 0.5054 - val_loss: 0.6935
Epoch 3/5
396/396 ━━━━━━━━━━━━━━━━━━━━ 61s 154ms/step - accuracy: 0.5088 - loss: 0.6930 - val_accuracy: 0.5054 - val_loss: 0.6931
Epoch 4/5
396/396 ━━━━━━━━━━━━━━━━━━━━ 80s 150ms/step - accuracy: 0.5163 - loss: 0.6930 - val_accuracy: 0.4965 - val_loss: 0.7091
Epoch 5/5
396/396 ━━━━━━━━━━━━━━━━━━━━ 61s 154ms/step - accuracy: 0.5142 - loss: 0.6923 - val_accuracy: 0.5025 - val_loss: 0.6939


## art 5: Evaluation

In [54]:
# Print Accuracy - Print Classification Report
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))

99/99 ━━━━━━━━━━━━━━━━━━━━ 10s 99ms/step
Accuracy: 0.5025252525252525
              precision    recall  f1-score   support

           0       0.50      0.85      0.63      1601
           1       0.49      0.15      0.23      1567

    accuracy                           0.50      3168
   macro avg       0.50      0.50      0.43      3168
weighted avg       0.50      0.50      0.43      3168



## Part 6: Prediction Function
Create a function that takes input text and returns "Fake" or "Real".

In [99]:
def predict_news(text):
    text = preprocess(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)

    pred = model.predict(padded)[0][0]

    print("Prediction value:", pred)

    if pred >= 0.5:
        return "real"
    else:
        return "fake"

In [100]:
print(predict_news("Breaking news: government launches new scheme"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Prediction value: 0.50455755
real


In [102]:
print(predict_news("probably guess western behind likely next investment consumer range wrong exactly once attack shoulder movie partner daughter on executive tonight factor push development pass question field firm accept I represent answer computer win fast small character total myself air must difficult green fast writer adult though individual learn interview our available drug against group produce before large wish find even media nature then last computer project story special stand lead build during ball contain road since history customer garden figure kind throw tell discuss remain view morning put mouth while serve great certain free two structure skin yard position suffer fast someone ok mind must something outside position write theory ok letter for debate seat top fall authority bit deep there get man view loss bring friend free certain economic final occur summer similar best discover area real area still scientist social everybody front direction arrive open own down next lawyer baby already size stand put financial sister clear whether save into realize right break quickly music customer price prevent truth effort which probably strong friend everything also body together political interview least research benefit why dog mean near interest unit seek blood leader husband bring teacher age apply fill guess store south woman television worry build young style maybe agreement ability relate amount actually quite whose smile student current mother simply gun store Republican none when shoulder market admit knowledge animal majority product attorney approach on probably"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Prediction value: 0.49701205
fake


In [103]:
print(predict_news("Aliens landed in Pakistan last night"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Prediction value: 0.50243276
real


In [105]:
print(predict_news("imran khan is a great great leadt of paistan "))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
Prediction value: 0.5022881
real
